# Hydrocarbon Porosity Predictor - Exploratory Data Analysis

This notebook provides exploratory data analysis for the well-log dataset used in the Hydrocarbon Porosity Predictor project.

## Objectives
- Load and explore the well-log data
- Analyze feature distributions
- Understand correlations between features and porosity
- Visualize data patterns

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 2. Load Data

In [ ]:
# Load the data
df = pd.read_csv('../data/sample_data.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

print("\nData Types:")
print(df.dtypes)

print("\nBasic Statistics:")
print(df.describe())

## 3. Data Quality Check

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Check for outliers (using IQR method)
print("\nOutlier Analysis (IQR Method):")
for column in df.select_dtypes(include=[np.number]).columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[column] < (Q1 - 1.5 * IQR)) | (df[column] > (Q3 + 1.5 * IQR))).sum()
    print(f"{column}: {outliers} outliers")

## 4. Feature Distributions

In [ ]:
# Plot distributions of all numeric features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

numeric_columns = df.select_dtypes(include=[np.number]).columns

for idx, column in enumerate(numeric_columns):
    axes[idx].hist(df[column], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {column}')
    axes[idx].set_xlabel(column)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)

# Hide empty subplot
axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

print("✓ Distribution plots displayed")

## 5. Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = df.corr()

print("Correlation Matrix:")
print(correlation_matrix)

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Well-Log Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Display correlations with target variable
print("\nCorrelations with Porosity (Target Variable):")
porosity_corr = df.corr()['Porosity'].sort_values(ascending=False)
print(porosity_corr)

## 6. Scatter Plots - Feature vs Target

In [ ]:
# Scatter plots of features vs porosity
features = ['GR', 'RT', 'NPHI', 'RHOB']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, feature in enumerate(features):
    axes[idx].scatter(df[feature], df['Porosity'], alpha=0.6, s=100, color='steelblue', edgecolors='navy')
    
    # Add trend line
    z = np.polyfit(df[feature], df['Porosity'], 1)
    p = np.poly1d(z)
    axes[idx].plot(df[feature], p(df[feature]), "r--", alpha=0.8, linewidth=2, label='Trend')
    
    axes[idx].set_xlabel(feature, fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Porosity', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{feature} vs Porosity', fontsize=12, fontweight='bold')
    axes[idx].grid(alpha=0.3)
    axes[idx].legend()
    
    # Calculate and display correlation
    corr = df[feature].corr(df['Porosity'])
    axes[idx].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                   transform=axes[idx].transAxes, 
                   fontsize=10, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("✓ Feature vs Target scatter plots displayed")

## 7. Box Plots - Feature Statistics

In [ ]:
# Box plots for each feature
fig, axes = plt.subplots(1, 5, figsize=(16, 5))

numeric_columns = ['GR', 'RT', 'NPHI', 'RHOB', 'Porosity']

for idx, column in enumerate(numeric_columns):
    axes[idx].boxplot(df[column], vert=True)
    axes[idx].set_ylabel(column, fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{column} Distribution', fontsize=11, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Box plots displayed")

## 8. Summary Statistics

In [ ]:
print("\n" + "="*60)
print("DATA SUMMARY")
print("="*60)

print(f"\nTotal Samples: {len(df)}")
print(f"Total Features: {len(df.columns)}")
print(f"Feature Names: {', '.join(df.columns.tolist())}")

print("\nTarget Variable Statistics (Porosity):")
print(f"  Mean: {df['Porosity'].mean():.4f}")
print(f"  Std Dev: {df['Porosity'].std():.4f}")
print(f"  Min: {df['Porosity'].min():.4f}")
print(f"  Max: {df['Porosity'].max():.4f}")
print(f"  Median: {df['Porosity'].median():.4f}")

print("\nKey Insights:")
porosity_corr = df.corr()['Porosity'].drop('Porosity').sort_values(key=abs, ascending=False)
for feature, corr in porosity_corr.items():
    strength = "Strong" if abs(corr) > 0.7 else "Moderate" if abs(corr) > 0.4 else "Weak"
    direction = "Positive" if corr > 0 else "Negative"
    print(f"  • {feature}: {strength} {direction} correlation ({corr:.3f})")

print("\n" + "="*60)